<a href="https://colab.research.google.com/github/niro2486/Statistical-Learning-e22093/blob/main/Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

## Sample Answer

### The 2PL Item Response Probability Model

##1.
Assuming that the item responses are conditionally independent given $\Theta=\theta$, the likelihood is

$$
P(Y=y\mid \Theta=\theta)=\prod_{i=1}^n
\left[
p_i(\theta)
\right]^{y_i}
\left[
1-p_i(\theta)
\right]^{1-y_i},
$$

where

$$
p_i(\theta)
=\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
# Two distinct a_i values (0.5 and 1.5)
# For a_i = 1.5, we include 3 different b_i values (-2, 0, 2)
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]

    # Calculate probabilities
    p_vals = p_i(theta_vals, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

# Customize layout
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Display the interactive plot
fig.show()


### 2. Sequential Likelihood Contribution
The likelihood contribution of a single new response $y_k \in \{0, 1\}$ at step $k$, given the latent ability $\theta$, is an independent Bernoulli trial:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Because each response is conditionally independent given the user's ability $\theta$, the joint likelihood function for the running history vector $y^{(k)}$ is the product of the individual likelihoods up to step $k$:

$$L(y^{(k)} \mid \theta) = \prod_{j=1}^{k} [p_j(\theta)]^{y_j} [1 - p_j(\theta)]^{1 - y_j}$$

---

### 3. Mathematical Formulation of the Running Update
Using Bayes' theorem sequentially, the posterior density at step $k$ is proportional to the product of the likelihood of the new observation and the posterior density from step $k-1$ (which acts as the new prior):

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

Substituting the Bernoulli likelihood:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

---

### 4. Dynamic Shifting
If the user answers a highly difficult item correctly ($y_k = 1$ and $b_k$ is large), the likelihood function $L(y_k=1 \mid \theta) = p_k(\theta)$ is near $0$ for low values of $\theta$ and asymptotically approaches $1$ only for high values of $\theta$.

When multiplying the running prior distribution by this likelihood function, the left tail of the prior is heavily penalized (multiplied by near-zero), while the right tail is preserved (multiplied by near-one). Mathematically, this shifts the mass of the distribution toward the right, resulting in a new posterior mode (peak) that is located at a higher $\theta$ value than the previous step.

---

### 5. Tracking Certainty and Sharpness
The discrimination parameter $a_k$ controls the steepness of the likelihood function.

* **Large $a_k$:** The logistic curve is very steep. The likelihood provides a sharp gradient separating abilities above $b_k$ from those below. When multiplied by the prior, it drastically cuts off one side of the distribution, significantly reducing the variance (sharpening the density) and increasing the platform's certainty about the user's ability bound.
* **Small $a_k$:** The logistic curve is flat. The likelihood function is relatively uniform across a wide range of $\theta$. Multiplying the prior by this flat likelihood provides little new information, resulting in minimal reduction in variance and leaving the posterior distribution broad and uncertain.

---

### 6. Numerical Implementation of a Running Grid
Because the posterior density does not have a closed-form solution under the 2PL model, it can be tracked numerically using grid approximation:

1. **Initialization:** Define a fixed grid of $M$ equally spaced points for $\theta$ (e.g., $-4$ to $4$ with $\Delta \theta = 0.01$). Evaluate the standard normal prior at each point to form an initial density array $P_0$.
2. **Sequential Update:** Upon observing $y_k$:
    * Evaluate the likelihood $L(y_k \mid \theta)$ for all $M$ points in the grid.
    * Multiply the likelihood array element-wise by the current prior array $P_{k-1}$ to get the unnormalized posterior $P_k^*$.
3. **Sequential Normalization:** To ensure the discrete array remains a valid probability density function, numerically integrate it using the Riemann sum. Calculate the sum of the array elements and multiply by the grid spacing $\Delta \theta$:
$$\text{Integral} \approx \sum_{j=1}^{M} P_k^*(\theta_j) \Delta \theta$$
Divide every element in $P_k^*$ by this scalar integral to yield the normalized posterior array $P_k$.

```

### Numerical Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

# 1. Define a fine grid of latent ability values (theta)
theta = np.linspace(-5, 5, 500)

# 2. Initialize the prior distribution: Standard Normal N(0, 1)
prior = stats.norm.pdf(theta, 0, 1)

# 3. Define the Item Response Function (2PL Model)
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 4. Define the manual simulated sequence of item encounters
running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},  # Step 1: Got an easy item correct
    {"a": 1.5, "b": 0.5,  "y": 1},  # Step 2: Got a medium-hard item correct
    {"a": 1.2, "b": 1.5,  "y": 0},  # Step 3: Got a very hard item incorrect
    {"a": 2.0, "b": 0.2,  "y": 1}   # Step 4: Got a highly discriminative item correct
]

# Create the first Plotly figure
fig1 = go.Figure()

# Add the initial Prior distribution trace
fig1.add_trace(go.Scatter(
    x=theta,
    y=prior,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Run the sequential update loop
current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a = item["a"]
    b = item["b"]
    y = item["y"]

    # Calculate item response probability across the grid
    prob = p_i(theta, a, b)

    # Calculate the likelihood contribution of this response
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))

    # Posterior proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerically normalize the density curve using the trapezoidal rule
    integral = np.trapezoid(current_posterior, theta)
    current_posterior /= integral

    # Format trace labels
    result_text = "Correct" if y == 1 else "Incorrect"
    trace_name = f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})"

    # Add the current posterior step to the plot
    fig1.add_trace(go.Scatter(
        x=theta,
        y=current_posterior,
        mode='lines',
        name=trace_name,
        line=dict(width=2)
    ))

# Customize layout for Figure 1
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the first plot
fig1.show()


# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

# Set random seed for reproducibility of the random items/responses
np.random.seed(42)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)  # Finer grid for tracking accuracy

# 2. Generate Random Item Characteristics
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 3. Initialize Tracking Arrays (Step 0 = Initial Prior State)
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior density distribution: N(0, 1)
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

# 4. Run the 20-item Sequential Simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Compute true response probability at true theta
    prob_true = p_i(theta_true, a_k, b_k)

    # Simulate stochastically generated user response y_k
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Calculate likelihood curve across grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    # Update and normalize
    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    # Compute point estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    # Store estimates
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 5. Create the second Plotly figure
fig2 = go.Figure()

# Add True Ability Reference Horizontal Line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})",
    annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Customize Layout for Figure 2
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

# Display the second plot
fig2.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### Visualization of the Beta Distribution

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Generate a dense grid of points over the bounded domain [0, 1]
theta_grid = np.linspace(0, 1, 500)

# 2. Define the three distinct parameter configurations
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

# 3. Create the Plotly figure
fig = go.Figure()

for config in beta_configs:
    a = config["alpha"]
    b = config["beta"]

    # Calculate the exact analytical probability density function (PDF)
    pdf_vals = stats.beta.pdf(theta_grid, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_grid,
        y=pdf_vals,
        mode='lines',
        name=config["name"],
        line=dict(color=config["color"], dash=config["dash"], width=2.5)
    ))

# 4. Customize layout and design
fig.update_layout(
    title={
        'text': "Structural Variations of the Beta(α, β) Probability Density Function",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Parameter Value (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the interactive plot
fig.show()

### 2. Sequential Likelihood and Joint History
The likelihood contribution of a single new interaction $y_k \in \{0, 1\}$ at step $k$, conditional on the true conversion rate $\theta$, is simply a Bernoulli trial:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Because each user impression is conditionally independent given $\theta$, the joint likelihood function for the running history vector $y^{(k)}$ is the product of the individual step likelihoods:

$$L(y^{(k)} \mid \theta) = \prod_{j=1}^{k} \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum_{j=1}^k y_j} (1 - \theta)^{k - \sum_{j=1}^k y_j}$$

---

### 3. Closed-Form Analytical Updates (Conjugacy)
Using Bayes' Theorem, the running posterior distribution is proportional to the product of the new likelihood and the previous step's posterior (the new prior):

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

Substituting the Bernoulli likelihood and the general Beta density function:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

By grouping the terms with the common bases $\theta$ and $(1-\theta)$ and adding their exponents:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This resulting expression is exactly the kernel (the variable portion) of a new Beta distribution. This mathematical harmony proves **conjugacy**—the posterior remains in the exact same distribution family as the prior.

The closed-form sequential updates for the shape parameters are simply arithmetic additions:
* $\alpha_k = \alpha_{k-1} + y_k$
* $\beta_k = \beta_{k-1} + (1 - y_k)$

The Posterior Mean at time step $k$ is calculated as the expected value of this new Beta distribution:
$$E[\Theta \mid Y^{(k)} = y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

---

### 4. Dynamic Shifting Mechanics
* **Observed Click ($y_k = 1$):** Mathematically, $y_k=1$ increments the $\alpha_k$ parameter by $1$ while leaving $\beta_k$ unchanged. This increases the ratio of $\alpha$ to $\beta$, physically shifting the peak of the probability density to the right (towards a higher probability of success).
* **Observed Non-Click ($y_k = 0$):** Conversely, $y_k=0$ increments the $\beta_k$ parameter by $1$. This increases the failure weight, shifting the peak of the density to the left.

**Contrast with Non-Conjugate Setups:**
In a conjugate model like Beta-Bernoulli, updating the platform's belief requires zero calculus or numerical approximation—it is achieved by adding $1$ to a counter. In a non-conjugate model, there is no closed-form solution. The system cannot just update two variables; it must instead maintain a massive numerical grid, calculate the new density at every single point on that grid, and computationally integrate the area under the curve to normalize it at every single step.

---

### 5. Running Point Estimators
At any step $k$, the platform can instantaneously extract these point estimates using the running $\alpha_k$ and $\beta_k$ parameters:

**Running Posterior Mean:**
$$\hat{\theta}^{(k)}_{Bayes} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**Running Maximum A Posteriori (MAP) (The Mode):**
*(Valid when $\alpha_k, \beta_k > 1$)*
$$\hat{\theta}^{(k)}_{MAP} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

### Numerical Simulation


In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.35  # The true click-through rate of the advertisement (35%)
n_impressions = 100
steps = list(range(n_impressions + 1))

# Initialize Beta Prior Parameters (Prior: Uniform distribution Beta(1,1))
alpha_param = 1
beta_param = 1

# Setup grid for plotting probability density curves (Domain: 0 to 1)
theta_grid = np.linspace(0, 1, 500)

# Define specific milestone steps where we want to capture the full curve shape
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

# Tracking arrays for point estimates
running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]  # Mode of Uniform(0,1)

# Initialize Figure 1 for the probability density progression curves
fig1 = go.Figure()

# Plot the initial Prior Beta(1, 1) distribution curve
prior_density = stats.beta.pdf(theta_grid, alpha_param, beta_param)
fig1.add_trace(go.Scatter(
    x=theta_grid, y=prior_density, mode='lines',
    name='Initial Prior: Beta(1,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# RUNNING ANALYTICAL UPDATE LOOP
# =====================================================================
for k in range(1, n_impressions + 1):
    # Simulate a user action stochastically based on true CTR
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # EXACT CLOSED-FORM CONJUGATE UPDATE RULES:
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Calculate exact point estimates directly from analytical formulas
    theta_bayes_k = alpha_param / (alpha_param + beta_param)

    # Guard formula for mode when alpha or beta <= 1
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    # If the current step hits one of our milestone checkpoints, capture its PDF
    if k in milestones:
        # Evaluate exact density curve across the grid analytically
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No Click"

        fig1.add_trace(go.Scatter(
            x=theta_grid, y=density_k, mode='lines',
            name=f"Step {k}: After Event ({result_text}, α={alpha_param}, β={beta_param})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE FIGURE 1: POSTERIOR DISTRIBUTION PROGRESSION
# =====================================================================
# Add True CTR vertical line to see where the density curves are peaking
fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2,
    annotation_text=f"True CTR ({theta_true})", annotation_position="top right"
)

fig1.update_layout(
    title={
        'text': "Analytical Posterior Density Progression (Beta-Binomial Updates)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
fig1.show()

# =====================================================================
# VISUALIZE FIGURE 2: CONVERGENCE TIMELINE
# =====================================================================
fig2 = go.Figure()

# Add True CTR Reference Line
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines',
    name='Exact Posterior Mean (Beta Formula)',
    line=dict(color='blue', width=2.5)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines',
    name='Exact MAP Estimate (Beta Formula)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()